In [18]:
import os
from typing import TypedDict, List, Union
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [19]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]

In [20]:
load_dotenv()

True

In [21]:
MODEL = "gpt-5-mini"

llm = ChatOpenAI(model=MODEL)

In [22]:
def process_llm(state: AgentState) -> AgentState:
    """
    This is a node that processes the LLM client
    """

    response = llm.invoke(state["messages"])
    print("AI: ", response.content)
    state["messages"].append(AIMessage(content=response.content))

    return state

In [23]:
agent = StateGraph(AgentState)

agent.add_node("llm_processor", process_llm)
agent.add_edge(START, "llm_processor")
agent.add_edge("llm_processor", END)

app = agent.compile()

In [27]:
conversation_history = []
user_input = input("User: ")

while user_input.lower() not in ["exit", "bye"]:
    conversation_history.append(HumanMessage(content=user_input))
    result = app.invoke(
        {
            "messages": conversation_history
        }
    )
    ### Update the conversation_list with the latest AIMessage, each turn.
    conversation_history = result["messages"]
    user_input = input("User: ")

User:  hey


AI:  Hey! How can I help you today? Quick examples of things I can do: answer questions, draft or edit text, debug code, summarize articles, help plan trips or meals, brainstorm ideas, or tutor you on a topic. What do you need?


User:  how are you?


AI:  I'm doing well, thanks — ready to help. How are you, and what can I do for you today?


User:  I don't know


AI:  That's okay — not always easy to put into words. Do you want to talk about it? I can:

- Listen if you want to vent/talk things through.
- Help figure out what’s bothering you (questions to sort it out).
- Suggest quick mood-boosters (breathing, short walk, music, a small task).
- Distract you (jokes, trivia, a game, or a story).
- Help with something practical (draft a message, plan, or to-do list).

If you’re feeling overwhelmed or in crisis and might hurt yourself, please reach out to local emergency services or a crisis line (in the U.S. call or text 988). What would you like to do right now?


User:  exit


In [28]:
with open("logging.txt", "w") as f:
    f.write("Your Conversation Log:\n\n") 

    for message in conversation_history:
        if isinstance(message, HumanMessage):
            f.write(f"You: {message.content}\n")
        elif isinstance(message, AIMessage):
            f.write(f"AI: {message.content}\n")

    f.write("\nEnd of Conversation")

print("Conversation saved to logging.txt")

Conversation saved to logging.txt
